# Geste Musique — entraînement sur Google Colab

Ce carnet entraîne les deux modèles TensorFlow de Geste Musique puis les exporte pour TensorFlow.js :

* **Modèle A — poses statiques** : petit MLP sur les 21 points normalisés (63 valeurs), pré-entraîné sur **HaGRID** puis affiné sur vos données.
* **Modèle B — gestes dynamiques** : 1D-CNN sur des séquences de 24 images × 67 valeurs (Jester, IPN Hand ou vos séquences).

👉 Menu *Exécution → Modifier le type d'exécution → GPU T4* (le modèle A tourne aussi très bien sur CPU).

## 1. Récupérer le projet et installer les dépendances

In [ ]:
# Remplacez par l'URL de votre dépôt (ou téléversez le dossier du projet dans Colab).
!git clone https://github.com/VOTRE-COMPTE/geste-musique.git
%cd geste-musique/training
!pip install -q mediapipe opencv-python-headless scikit-learn  # tensorflow et tf_keras sont déjà sur Colab

In [ ]:
# Vérifie que la normalisation Python est identique à celle du navigateur.
!python -m pytest -q test_common.py

## 2. Télécharger HaGRID (poses statiques)

Le plus simple : l'échantillon de 30 000 images en 384 px publié sur Kaggle.
Créez un jeton API sur kaggle.com (*Settings → API → Create New Token*) et téléversez `kaggle.json`.
Pour le jeu complet (HaGRID v1 ou v2, ~1 million d'images), suivez https://github.com/hukenovs/hagrid.

In [ ]:
from google.colab import files
files.upload()  # choisissez kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d innominate817/hagrid-sample-30k-384p -p data/ --unzip
!ls data/

## 3. Extraire les points avec MediaPipe (≈ 20–40 min pour 30 000 images)

In [ ]:
# Adaptez les chemins à ce que montre `ls` ci-dessus.
!python extract_hagrid.py \
    --images data/hagrid-sample-30k-384p/hagrid_30k \
    --annotations data/hagrid-sample-30k-384p/ann_train_val \
    --out data/hagrid_landmarks.npz --max-per-class 2000

## 4. Vos données

Dans l'application : onglet **Données** → enregistrez des poses (et des séquences) pour plusieurs personnes,
puis **Exporter le JSON** et téléversez-le ici. C'est indispensable : le modèle doit voir *votre* caméra et *vos* mains,
et une classe `none` (mains au repos, transitions) pour éviter les faux déclenchements.

In [ ]:
uploaded = files.upload()  # dataset-geste-musique-....json
MY_DATA = 'data/' + list(uploaded)[0]
!mv "{list(uploaded)[0]}" data/

## 5. Entraîner le modèle A (pré-entraînement HaGRID + affinage sur vos données)

In [ ]:
!python train_pose.py --npz data/hagrid_landmarks.npz --finetune-json "$MY_DATA" --epochs 60

## 6. (Optionnel) Modèle B — gestes dynamiques

* **Jester** (148 000 vidéos, inscription sur le site de Qualcomm) : `python extract_videos.py jester --frames … --csv …`
* **IPN Hand** ou vos vidéos rangées par dossier de classe : `python extract_videos.py folders --root …`
* ou seulement les séquences enregistrées dans l'application :

In [ ]:
!python train_dynamic.py --json "$MY_DATA" --epochs 60

## 7. Récupérer les modèles pour l'application

Décompressez l'archive dans `public/models/` du projet : l'application les charge au démarrage (onglet Réglages → Modèles).

In [ ]:
!cd ../public && zip -r /content/modeles-geste-musique.zip models/pose models/dynamic 2>/dev/null; ls -la /content/*.zip
files.download('/content/modeles-geste-musique.zip')